In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gradio -q

import gradio as gr
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

base = '/content/drive/MyDrive/KemikAI'
model = keras.models.load_model(f'{base}/models/kemikai_multimodal_best.keras')
print("✅ Model ve Gradio hazır!")

Mounted at /content/drive
✅ Model ve Gradio hazır!


In [2]:
# Scaler hazırla
df = pd.read_csv('/content/drive/MyDrive/KemikAI/data/raw/RSNA_Annotations/RSNA_Annotations/BONEAGE/boneage_train.csv')
df['gender'] = df['Male'].apply(lambda x: 1 if x else 0)

age_y = df['Boneage'] / 12.0
df['height_cm'] = np.random.normal(70 + (age_y * 6), 5).clip(50, 200)
df['weight_kg'] = np.random.normal(10 + (age_y * 2.5), 3).clip(3, 120)
df['mother_height'] = np.random.normal(162, 6, size=len(df)).clip(145, 185)
df['father_height'] = np.random.normal(175, 7, size=len(df)).clip(160, 195)

continuous_cols = ['height_cm', 'weight_kg', 'mother_height', 'father_height']
scaler = StandardScaler()
scaler.fit(df[continuous_cols])

print("✅ Scaler hazır!")

✅ Scaler hazır!


In [3]:
def gradcam_heatmap(model, img_tensor, tabular_tensor, layer_name='top_activation'):
    tabular_tensor = tf.convert_to_tensor(tabular_tensor, dtype=tf.float32)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(
            {'image_input': img_tensor, 'tabular_input': tabular_tensor}
        )
        loss = predictions[0][0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def tahmin_yap(image, cinsiyet, boy, kilo, anne_boy, baba_boy,
               vit_d, kalsiyum, kirik, kronolojik_ay):

    if image is None:
        return "Lütfen röntgen görüntüsü yükleyin.", None

    # Görüntü işleme
    img = tf.image.resize(image, (224, 224))
    img = tf.cast(img, tf.float32)
    if img.shape[-1] == 1:
        img = tf.image.grayscale_to_rgb(img)
    elif img.shape[-1] == 4:
        img = img[:, :, :3]
    img_batch = tf.expand_dims(img, axis=0)

    # Tabular veri
    gender = 1.0 if cinsiyet == "Erkek" else 0.0
    vit_d_val = 1.0 if vit_d == "Var" else 0.0
    kirik_val = 1.0 if kirik == "Var" else 0.0
    kalsiyum_map = {"Düşük": 0, "Normal": 1, "Yüksek": 2}
    kalsiyum_val = float(kalsiyum_map[kalsiyum])

    continuous = np.array([[boy, kilo, anne_boy, baba_boy]])
    continuous_scaled = scaler.transform(continuous)
    categorical = np.array([[gender, vit_d_val, kalsiyum_val, kirik_val]])
    tabular = np.concatenate([continuous_scaled, categorical], axis=1).astype(np.float32)

    # Tahmin
    pred = model.predict(
        {'image_input': img_batch, 'tabular_input': tabular}, verbose=0
    )[0][0]
    pred_yil = pred / 12

    # Gelişim durumu
    if kronolojik_ay > 0:
        fark = pred - kronolojik_ay
        if abs(fark) <= 6:
            durum = "✅ Normal Gelişim"
        elif fark < -6:
            durum = f"⚠️ Gelişim Geride ({abs(fark):.0f} ay)"
        else:
            durum = f"📈 Gelişim İleride ({fark:.0f} ay)"
    else:
        durum = "ℹ️ Kronolojik yaş girilmedi"

    # GradCAM
    heatmap = gradcam_heatmap(model, img_batch, tabular)
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    orig = img.numpy().astype(np.uint8)
    overlay = cv2.addWeighted(orig, 0.6, heatmap_color, 0.4, 0)

    # Rapor
    rapor = f"""
🦴 KemiKAI — Gelişim Raporu
─────────────────────────────
📊 Tahmini Kemik Yaşı : {pred:.0f} ay ({pred_yil:.1f} yıl)
👤 Cinsiyet           : {cinsiyet}
📏 Boy / Kilo         : {boy} cm / {kilo} kg
👨‍👩‍👧 Anne / Baba Boyu   : {anne_boy} cm / {baba_boy} cm
💊 D Vitamini         : {vit_d}
🧪 Kalsiyum           : {kalsiyum}
🦴 Kırık Geçmişi      : {kirik}
─────────────────────────────
📈 Gelişim Durumu     : {durum}
─────────────────────────────
⚕️ Bu sonuç yalnızca bilgilendirme
amaçlıdır. Kesin tanı için uzman
hekime başvurunuz.
    """

    return rapor, overlay

print("✅ Fonksiyon hazır!")

✅ Fonksiyon hazır!


In [ ]:
with gr.Blocks(title="KemiKAI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🦴 KemiKAI — Çocuklarda Kemik Gelişim Analizi")
    gr.Markdown("El röntgeni ve klinik bilgilerle kemik yaşı tahmini yapın.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="El Röntgeni Yükle", type="numpy")
            cinsiyet = gr.Radio(["Kız", "Erkek"], label="Cinsiyet", value="Kız")
            kronolojik_ay = gr.Slider(0, 228, value=0, step=1, label="Kronolojik Yaş (ay) — 0 ise atla")

            with gr.Row():
                boy = gr.Slider(50, 200, value=120, step=1, label="Boy (cm)")
                kilo = gr.Slider(3, 120, value=30, step=1, label="Kilo (kg)")

            with gr.Row():
                anne_boy = gr.Slider(145, 185, value=162, step=1, label="Anne Boyu (cm)")
                baba_boy = gr.Slider(160, 195, value=175, step=1, label="Baba Boyu (cm)")

            with gr.Row():
                vit_d = gr.Radio(["Yok", "Var"], label="D Vitamini Eksikliği", value="Yok")
                kalsiyum = gr.Radio(["Düşük", "Normal", "Yüksek"], label="Kalsiyum Seviyesi", value="Normal")
                kirik = gr.Radio(["Yok", "Var"], label="Kırık Geçmişi", value="Yok")

            btn = gr.Button("🔍 Analiz Et", variant="primary")

        with gr.Column():
            rapor_output = gr.Textbox(label="📋 Gelişim Raporu", lines=15)
            gradcam_output = gr.Image(label="🔬 GradCAM Görselleştirme")

    btn.click(
        fn=tahmin_yap,
        inputs=[image_input, cinsiyet, boy, kilo, anne_boy, baba_boy,
                vit_d, kalsiyum, kirik, kronolojik_ay],
        outputs=[rapor_output, gradcam_output]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_27111/696593738.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="KemiKAI", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7842797e9b139f01ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
